# Player Stats — Cleaned Pipeline

Re-aggregates player club stats from raw Transfermarkt performances.

**Fixes applied:**
- Replaced broken `minutes_played` with `nb_on_pitch` (appearances on pitch)
- Domestic league identified via `competition_id` mapping (not `competition_name`)
- Stats summed across ALL competitions per season, not just primary
- Per-appearance normalization added
- Coverage flags for downstream filtering

In [20]:
import pandas as pd
import re
from rapidfuzz import process, fuzz
from unidecode import unidecode
import os
from dotenv import load_dotenv

## 1. Load Data

In [21]:
load_dotenv()

TRANSFERMARKT_DIR = os.getenv("TRANSFERMARKT_DIR")

roster = pd.read_csv("player_rosters.csv")
profiles = pd.read_csv(os.path.join(TRANSFERMARKT_DIR, "player_profiles.csv"))
performances = pd.read_csv(os.path.join(TRANSFERMARKT_DIR, "player_performances.txt"))
market_values = pd.read_csv(os.path.join(TRANSFERMARKT_DIR, "player_market_value.csv"))
tcs = pd.read_csv(os.path.join(TRANSFERMARKT_DIR, "team_competitions_seasons.csv"))

print(f"Roster: {roster.shape}")
print(f"Profiles: {profiles.shape}")
print(f"Performances: {performances.shape}")

/var/folders/2_/skwfrb393zjdzz43_1hyfg8m0000gq/T/ipykernel_5612/2862854227.py:6: DtypeWarning: Columns (29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  profiles = pd.read_csv(os.path.join(TRANSFERMARKT_DIR, "player_profiles.csv"))


Roster: (5550, 25)
Profiles: (92671, 34)
Performances: (1878719, 20)


## 2. Filter & Map Seasons

In [22]:
WC_PRIOR_SEASON = {
    "WC-2006": "05/06",
    "WC-2010": "09/10",
    "WC-2014": "13/14",
    "WC-2018": "17/18",
    "WC-2022": "21/22",
}
roster = roster[roster["tournament_id"].isin(WC_PRIOR_SEASON)].copy()
roster["target_season"] = roster["tournament_id"].map(WC_PRIOR_SEASON)

print(f"Roster filtered: {roster.shape}")
print(roster["tournament_id"].value_counts().sort_index())

Roster filtered: (3775, 26)
tournament_id
WC-2006    736
WC-2010    736
WC-2014    736
WC-2018    736
WC-2022    831
Name: count, dtype: int64


## 3. Name Matching (Roster → Transfermarkt)

Six-stage fuzzy matching pipeline using DOB anchoring + progressively relaxed name thresholds.

In [23]:
NICKNAME_MAP = {
    "juninho pernambucano": "amarildo tavares da silveira",
    "kaka": "ricardo izecson dos santos leite",
    "ronaldo": "ronaldo luis nazario de lima",
    "ronaldinho": "ronaldo de assis moreira",
    "rivaldo": "rivaldo vitor borba ferreira",
    "adriano": "adriano leite ribeiro",
    "fred": "frederico chaves guedes",
    "hulk": "givanildo vieira de sousa",
    "neymar": "neymar da silva santos junior",
    "robinho": "robinson de souza",
    "diego": "diego ribas da cunha",
    "cafu": "marcos evangelista de morais",
    "roberto carlos": "roberto carlos da silva",
    "bebeto": "jose roberto gama de oliveira",
    "romario": "romario de souza faria",
    "emerson": "emerson ferreira da rosa",
    "gilberto silva": "gilberto aparecido da silva",
    "lucio": "lucimar ferreira da silva",
    "maicon": "maicon douglas sisenando",
    "jo": "joao alves de assis silva",
    "elano": "elano blumer",
    "grafite": "vander luis de souza",
    "pauleta": "pedro miguel pauleta",
    "maniche": "nuno ribeiro maniche",
    "deco": "anderson luis de abreu oliveira",
    "chicharito": "javier hernandez balcazar",
    "kun aguero": "sergio leonel aguero del castillo",
    "gio": "giovanni dos santos",
    "cuauhtemoc blanco": "cuauhtemoc blanco bravo",
    "el hadji diouf": "el hadji ousseynou diouf",
    "didier drogba": "didier yves drogba tebily",
    "yaya toure": "gnegneri yaya toure",
    "kolo toure": "souleymane kolo toure",
    "gervinho": "gervais lombe yao kouassi",
    "samuel etoo": "samuel etoo fils",
    "eto o": "samuel etoo fils",
    "geremi": "geremi sorele njitap fotso",
    "dado prso": "dado prso",
}


def norm(s):
    if not pd.notna(s) or str(s).strip() == "":
        return ""
    s = unidecode(str(s)).lower().strip()
    s = re.sub(r"[-'`\u2019]", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# Roster names
roster["full_name"] = (
    roster["given_name"].replace("not applicable", "").fillna("")
    + " "
    + roster["family_name"].fillna("")
).str.strip()
roster["full_name_norm"] = roster["full_name"].apply(norm)
roster["dob_clean"] = pd.to_datetime(roster["birth_date"], errors="coerce")

# TM profile names
profiles["player_name_clean"] = profiles["player_name"].str.replace(
    r"\s*\(\d+\)\s*$", "", regex=True
)
profiles["full_name_norm"] = profiles["player_name_clean"].apply(norm)
profiles["home_name_norm"] = profiles["name_in_home_country"].apply(norm)
profiles["dob_clean"] = pd.to_datetime(profiles["date_of_birth"], errors="coerce")

tm_index = profiles[
    ["player_id", "player_name_clean", "full_name_norm", "home_name_norm", "dob_clean"]
].copy()
tm_index = tm_index.dropna(subset=["full_name_norm"])
tm_index = tm_index[tm_index["full_name_norm"] != ""].reset_index(drop=True)

tm_names = tm_index["full_name_norm"].tolist()
tm_ids = tm_index["player_id"].tolist()
tm_home_names = tm_index["home_name_norm"].tolist()

print(f"TM index: {len(tm_index)} players")

TM index: 92151 players


In [24]:
def try_match_df(name, candidates_df, scorer, cutoff):
    if candidates_df.empty or not name:
        return None
    result = process.extractOne(
        name,
        candidates_df["full_name_norm"].tolist(),
        scorer=scorer,
        score_cutoff=cutoff,
    )
    if result:
        _, score, idx = result
        return int(candidates_df.iloc[idx]["player_id"]), score
    return None


matched_rows = []

for _, row in roster.iterrows():
    name = row["full_name_norm"]
    dob = row["dob_clean"]
    resolved = False
    search_name = NICKNAME_MAP.get(name, name)

    dob_candidates = pd.DataFrame()
    if pd.notna(dob):
        dob_candidates = tm_index[tm_index["dob_clean"] == dob].reset_index(drop=True)

    # S1: DOB + name (85)
    if not dob_candidates.empty:
        res = try_match_df(search_name, dob_candidates, fuzz.token_sort_ratio, 85)
        if res:
            matched_rows.append(
                {
                    **row.to_dict(),
                    "tm_player_id": res[0],
                    "match_score": res[1],
                    "match_method": "dob+name",
                }
            )
            resolved = True

    # S2: DOB + relaxed (70)
    if not resolved and not dob_candidates.empty:
        res = try_match_df(search_name, dob_candidates, fuzz.token_sort_ratio, 70)
        if res:
            matched_rows.append(
                {
                    **row.to_dict(),
                    "tm_player_id": res[0],
                    "match_score": res[1],
                    "match_method": "dob+relaxed",
                }
            )
            resolved = True

    # S3: DOB + last name only (92)
    if not resolved and not dob_candidates.empty:
        last = norm(row["family_name"])
        if last:
            dob_candidates_ln = dob_candidates.copy()
            dob_candidates_ln["last_norm"] = dob_candidates_ln["full_name_norm"].apply(
                lambda x: x.split()[-1] if x.strip() else ""
            )
            result = process.extractOne(
                last,
                dob_candidates_ln["last_norm"].tolist(),
                scorer=fuzz.ratio,
                score_cutoff=92,
            )
            if result:
                _, score, idx = result
                matched_rows.append(
                    {
                        **row.to_dict(),
                        "tm_player_id": int(dob_candidates_ln.iloc[idx]["player_id"]),
                        "match_score": score,
                        "match_method": "dob+lastname",
                    }
                )
                resolved = True

    # S4: DOB + home country name (85)
    if not resolved and not dob_candidates.empty:
        home_candidates = dob_candidates.copy()
        home_candidates["full_name_norm"] = home_candidates["home_name_norm"].apply(
            norm
        )
        home_candidates = home_candidates[
            home_candidates["full_name_norm"] != ""
        ].reset_index(drop=True)
        res = try_match_df(search_name, home_candidates, fuzz.token_sort_ratio, 85)
        if res:
            matched_rows.append(
                {
                    **row.to_dict(),
                    "tm_player_id": res[0],
                    "match_score": res[1],
                    "match_method": "dob+homename",
                }
            )
            resolved = True

    # S5: name-only fuzzy (92)
    if not resolved:
        result = process.extractOne(
            search_name, tm_names, scorer=fuzz.token_sort_ratio, score_cutoff=92
        )
        if result:
            _, score, idx = result
            matched_rows.append(
                {
                    **row.to_dict(),
                    "tm_player_id": int(tm_ids[idx]),
                    "match_score": score,
                    "match_method": "name_only",
                }
            )
            resolved = True

    # S6: home name only (92)
    if not resolved:
        result = process.extractOne(
            search_name, tm_home_names, scorer=fuzz.token_sort_ratio, score_cutoff=92
        )
        if result:
            _, score, idx = result
            matched_rows.append(
                {
                    **row.to_dict(),
                    "tm_player_id": int(tm_ids[idx]),
                    "match_score": score,
                    "match_method": "home_only",
                }
            )
            resolved = True

matched = pd.DataFrame(matched_rows)
print(matched["match_method"].value_counts())
print(f"\nMatch rate: {len(matched)}/{len(roster)} ({len(matched)/len(roster):.1%})")

match_method
dob+name        2702
name_only         96
dob+lastname      42
dob+relaxed       42
dob+homename       7
home_only          2
Name: count, dtype: int64

Match rate: 2891/3775 (76.6%)


## 4. Domestic League Mapping

`competition_id` → standardized league name. Handles Apertura/Clausura splits, alternate IDs, and second divisions.

In [25]:
ALL_DOMESTIC = {
    # Big 5
    "GB1": "Premier League",
    "ES1": "LaLiga",
    "L1": "Bundesliga",
    "IT1": "Serie A",
    "FR1": "Ligue 1",
    # Europe tier 2
    "NL1": "Eredivisie",
    "PO1": "Liga Portugal",
    "TR1": "Super Lig",
    "RU1": "Premier Liga",
    "BE1": "JPL",
    "SC1": "Scottish Premiership",
    "GR1": "Super League Greece",
    "UKR1": "Ukrainian Premier Liga",
    "C1": "Super League Switzerland",
    "A1": "Austrian Bundesliga",
    "DK1": "Danish Superliga",
    "PL1": "Ekstraklasa",
    "SER1": "Super Liga Serbia",
    "SR1C": "Super Liga Serbia",
    "SRB1": "Super Liga Serbia",
    "HRV1": "1. HNL",
    "CZE1": "Czech First League",
    "TS1": "Czech First League",
    "ROM1": "Liga I Romania",
    "RO1": "Liga I Romania",
    "SLO1": "Slovak Super Liga",
    "UNG1": "NB I Hungary",
    "BU1": "Bulgarian First League",
    "ZYP1": "Cypriot First Division",
    "SWE1": "Allsvenskan",
    "NOR1": "Eliteserien",
    "ISR1": "Ligat haAl",
    "LUX1": "BGL Ligue",
    # Americas
    "MLS1": "MLS",
    "MEX1": "Liga MX",
    "MEXA": "Liga MX",
    "ARGC": "Argentine Primera",
    "ARG1": "Argentine Primera",
    "AR1N": "Argentine Primera",
    "CRPD": "Costa Rica Primera",
    "PDV1": "Paraguay Primera",
    "HO1A": "Liga Nacional Honduras",
    "HOC1": "Liga Nacional Honduras",
    "GU1A": "Liga Nacional Guatemala",
    "GU1C": "Liga Nacional Guatemala",
    # Asia / Africa / Oceania
    "AUS1": "A-League",
    "JAP1": "J1 League",
    "KR1": "K League 1",
    "KLS1": "K League 1",
    "KLS2": "K League 1",
    "SA1": "Saudi Pro League",
    "KSA1": "Saudi Pro League",
    "QSL": "Qatar Stars League",
    "IRN1": "Persian Gulf League",
    "SFA1": "South African Premiership",
    "EGY1": "Egyptian Premier League",
    "MAR1": "Botola Pro",
    "TUN1": "Ligue I Pro Tunisia",
    "NZL1": "NZ Football Championship",
    "BRA1": "Brasileirao",
    # Second / lower divisions
    "GB2": "Championship",
    "GB3": "League One",
    "GB4": "League Two",
    "SC2": "Scottish Championship",
    "L2": "2. Bundesliga",
    "FR2": "Ligue 2",
    "IT2": "Serie B",
    "ES2": "LaLiga 2",
    "PO2": "Liga Portugal 2",
    "NL2": "Eerste Divisie",
    "BE2": "Belgian Second Division",
    "DK2": "Danish First Division",
    "TR2": "Turkish 1. Lig",
}

print(f"Domestic league map: {len(ALL_DOMESTIC)} competition_id entries")

Domestic league map: 73 competition_id entries


## 5. Re-aggregate from Raw Performances

For each player-tournament pair:
- Identify domestic league via `competition_id` (not `competition_name`)
- Label primary club from highest-apps domestic competition
- Sum stats across ALL competitions (domestic + cups + continental)
- Normalize per appearance using `nb_on_pitch`

In [26]:
# Join matched roster with raw performances on (player_id, season)
merged = performances.merge(
    matched[["tm_player_id", "tournament_id", "target_season"]].rename(
        columns={"tournament_id": "wc_tournament"}
    ),
    left_on=["player_id", "season_name"],
    right_on=["tm_player_id", "target_season"],
    how="inner",
)
merged["domestic_league"] = merged["competition_id"].map(ALL_DOMESTIC)

print(f"Merged performance rows: {len(merged)}")
print(f"Unique players matched: {merged['tm_player_id'].nunique()}")

Merged performance rows: 9139
Unique players matched: 1981


In [27]:
# ── Domestic league primary club ────────────────────────────────────────────
dom = merged[merged["domestic_league"].notna()].sort_values(
    "nb_on_pitch", ascending=False
)
dom_primary = dom.groupby(["tm_player_id", "wc_tournament"]).first().reset_index()
dom_primary = dom_primary[
    ["tm_player_id", "wc_tournament", "domestic_league", "team_name", "competition_id"]
]
dom_primary.columns = [
    "tm_player_id",
    "wc_tournament",
    "home_league",
    "primary_club",
    "primary_comp_id",
]

# ── Fallback for cup-only players ──────────────────────────────────────────
fb = (
    merged.sort_values("nb_on_pitch", ascending=False)
    .groupby(["tm_player_id", "wc_tournament"])
    .first()
    .reset_index()
)
fb = fb[
    ["tm_player_id", "wc_tournament", "competition_name", "team_name", "competition_id"]
]
fb.columns = [
    "tm_player_id",
    "wc_tournament",
    "fallback_league",
    "fallback_club",
    "fallback_comp_id",
]

print(f"With domestic league: {len(dom_primary)}")
print(
    f"Total player-tournaments: {merged.groupby(['tm_player_id','wc_tournament']).ngroups}"
)

With domestic league: 2552
Total player-tournaments: 2665


In [28]:
# ── Aggregate ALL competitions per player per tournament ────────────────────
agg = (
    merged.groupby(["tm_player_id", "wc_tournament"])
    .agg(
        total_appearances=("nb_on_pitch", "sum"),
        total_goals=("goals", "sum"),
        total_assists=("assists", "sum"),
        total_own_goals=("own_goals", "sum"),
        total_penalty_goals=("penalty_goals", "sum"),
        total_yellow_cards=("yellow_cards", "sum"),
        total_second_yellows=("second_yellow_cards", "sum"),
        total_red_cards=("direct_red_cards", "sum"),
        total_goals_conceded=("goals_conceded", "sum"),
        total_clean_sheets=("clean_sheets", "sum"),
        num_competitions=("competition_id", "nunique"),
    )
    .reset_index()
)

# Join domestic info + fallback
result = agg.merge(dom_primary, on=["tm_player_id", "wc_tournament"], how="left")
result = result.merge(fb, on=["tm_player_id", "wc_tournament"], how="left")

result["club"] = result["primary_club"].fillna(result["fallback_club"])
result["league"] = result["home_league"].fillna(result["fallback_league"])

result = result.drop(
    columns=[
        "primary_club",
        "primary_comp_id",
        "fallback_league",
        "fallback_club",
        "fallback_comp_id",
    ]
)

print(f"Aggregated rows: {len(result)}")

Aggregated rows: 2665


## 6. Per-Appearance Normalization

In [29]:
apps = result["total_appearances"].replace(0, float("nan"))

result["goals_per_app"] = result["total_goals"] / apps
result["assists_per_app"] = result["total_assists"] / apps
result["yellows_per_app"] = result["total_yellow_cards"] / apps
result["goals_conceded_per_app"] = result["total_goals_conceded"] / apps
result["clean_sheet_rate"] = result["total_clean_sheets"] / apps

result.head()

,tm_player_id,wc_tournament,total_appearances,total_goals,total_assists,total_own_goals,total_penalty_goals,total_yellow_cards,total_second_yellows,total_red_cards,...,total_clean_sheets,num_competitions,home_league,club,league,goals_per_app,assists_per_app,yellows_per_app,goals_conceded_per_app,clean_sheet_rate
0,10,WC-2006,40,31.0,17,0,1,5,1,0,...,0,5,Bundesliga,SV Werder Bremen,Bundesliga,0.775000,0.425000,0.125000,0.000000,0.000000
1,10,WC-2010,38,6.0,2,0,0,3,0,0,...,0,3,Bundesliga,Bayern Munich,Bundesliga,0.157895,0.052632,0.078947,0.000000,0.000000
2,10,WC-2014,29,8.0,5,0,0,2,0,0,...,0,4,Serie A,SS Lazio,Serie A,0.275862,0.172414,0.068966,0.000000,0.000000
3,14,WC-2006,6,1.0,0,0,0,0,0,0,...,0,2,Austrian Bundesliga,Red Bull Salzburg,Austrian Bundesliga,0.166667,0.000000,0.000000,0.000000,0.000000
4,26,WC-2014,43,0.0,0,0,0,1,0,0,...,13,4,Bundesliga,Borussia Dortmund,Bundesliga,0.000000,0.000000,0.023256,1.139535,0.302326


## 7. Rejoin Roster Metadata & Flag Coverage

In [30]:
roster_meta = matched[
    [
        "key_id",
        "tm_player_id",
        "tournament_id",
        "position_code",
        "family_name",
        "given_name",
        "team_name",
        "birth_date",
        "match_method",
        "match_score",
    ]
].rename(columns={"tournament_id": "wc_tournament", "team_name": "national_team"})

final = roster_meta.merge(result, on=["tm_player_id", "wc_tournament"], how="left")

final["has_stats"] = final["total_appearances"].notna() & (
    final["total_appearances"] > 0
)
final["has_domestic"] = final["home_league"].notna()

print(f"Shape: {final.shape}")
print(f"Has stats (apps > 0): {final['has_stats'].sum()}/{len(final)}")
print(f"Has domestic league: {final['has_domestic'].sum()}/{len(final)}")
print(f"Zero-app players: {(~final['has_stats']).sum()}")

Shape: (2891, 31)
Has stats (apps > 0): 2663/2891
Has domestic league: 2552/2891
Zero-app players: 228


## 8. Spot Checks

In [31]:
for name in ["Messi", "Ronaldo", "Mbappé", "Courtois", "Casillas"]:
    r = final[final["family_name"].str.contains(name, na=False)]
    if len(r):
        print(f"=== {name} ===")
        print(
            r[
                [
                    "wc_tournament",
                    "club",
                    "league",
                    "total_appearances",
                    "total_goals",
                    "goals_per_app",
                    "position_code",
                ]
            ].to_string()
        )
        print()

=== Messi ===
     wc_tournament                 club   league  total_appearances  total_goals  goals_per_app position_code
24         WC-2006         FC Barcelona   LaLiga               25.0          8.0       0.320000            FW
482        WC-2010         FC Barcelona   LaLiga               51.0         45.0       0.882353            FW
1015       WC-2014         FC Barcelona   LaLiga               46.0         41.0       0.891304            FW
1584       WC-2018         FC Barcelona   LaLiga               54.0         45.0       0.833333            FW
2179       WC-2022  Paris Saint-Germain  Ligue 1               34.0         11.0       0.323529            FW

=== Ronaldo ===
     wc_tournament               club          league  total_appearances  total_goals  goals_per_app position_code
52         WC-2006        Real Madrid          LaLiga               27.0         15.0       0.555556            FW
311        WC-2006  Manchester United  Premier League               47.0       

## 9. Player Market Value

Pre-tournament market value from Transfermarkt. For each player-tournament, we take the most recent valuation before the World Cup start date.

In [32]:
# ── Load market values ──────────────────────────────────────────────────────
market_values = pd.read_csv(os.path.join(TRANSFERMARKT_DIR, "player_market_value.csv"))
market_values["date"] = pd.to_datetime(market_values["date_unix"], errors="coerce")

print(f"Market values: {market_values.shape}")
print(market_values.head())

Market values: (901429, 4)
   player_id   date_unix       value       date
0    1000135  2023-12-19    100000.0 2023-12-19
1    1000135  2024-06-23    100000.0 2024-06-23
2         10  2005-01-06   9000000.0 2005-01-06
3         10  2008-06-03  20000000.0 2008-06-03
4         10  2009-08-29  12000000.0 2009-08-29


In [33]:
# World Cup start dates (approximate — before group stage)
WC_CUTOFF = {
    "WC-2006": "2006-06-09",
    "WC-2010": "2010-06-11",
    "WC-2014": "2014-06-12",
    "WC-2018": "2018-06-14",
    "WC-2022": "2022-11-20",
}

mv_rows = []
for _, row in final[final["tm_player_id"].notna()].iterrows():
    pid = int(row["tm_player_id"])
    wc = row["wc_tournament"]
    cutoff = pd.Timestamp(WC_CUTOFF[wc])

    player_mv = market_values[
        (market_values["player_id"] == pid) & (market_values["date"] <= cutoff)
    ]

    if len(player_mv):
        latest = player_mv.sort_values("date").iloc[-1]
        mv_rows.append(
            {
                "key_id": row["key_id"],
                "market_value": latest["value"],
                "mv_date": latest["date"],
            }
        )
    else:
        mv_rows.append(
            {
                "key_id": row["key_id"],
                "market_value": None,
                "mv_date": None,
            }
        )

mv_df = pd.DataFrame(mv_rows)
final = final.merge(mv_df, on="key_id", how="left")

print(f"Market value coverage: {final['market_value'].notna().sum()}/{len(final)}")
print(f"\nSample (top valued players):")
print(
    final[final["market_value"].notna()]
    .nlargest(10, "market_value")[
        ["family_name", "given_name", "wc_tournament", "club", "market_value"]
    ]
    .to_string()
)

Market value coverage: 2825/2891

Sample (top valued players):
     family_name      given_name wc_tournament                 club  market_value
1584       Messi          Lionel       WC-2018         FC Barcelona   180000000.0
1650      Neymar  not applicable       WC-2018  Paris Saint-Germain   180000000.0
2439      Mbappé          Kylian       WC-2022  Paris Saint-Germain   160000000.0
1624   De Bruyne           Kevin       WC-2018      Manchester City   150000000.0
1743       Salah         Mohamed       WC-2018         Liverpool FC   150000000.0
1756        Kane           Harry       WC-2018    Tottenham Hotspur   150000000.0
1015       Messi          Lionel       WC-2014         FC Barcelona   120000000.0
1780      Mbappé          Kylian       WC-2018  Paris Saint-Germain   120000000.0
2265      Júnior        Vinícius       WC-2022          Real Madrid   120000000.0
1593      Dybala           Paulo       WC-2018          Juventus FC   110000000.0


## 10. Club & League Rankings

From `team_competitions_seasons`: domestic league table position (`season_rank`) and league tier (`season_league_level_level_number`) for each player's primary club in the prior season.

In [34]:
# ── Load team competition seasons ───────────────────────────────────────────
tcs = pd.read_csv(os.path.join(TRANSFERMARKT_DIR, "team_competitions_seasons.csv"))

print(f"Team competitions seasons: {tcs.shape}")
print(tcs.columns.tolist())

Team competitions seasons: (58247, 29)
['club_division', 'club_id', 'competition_id', 'competition_name', 'season_draws', 'season_goal_difference', 'season_goals_against', 'season_goals_for', 'season_id', 'season_is_two_point_system', 'season_league_competition_id', 'season_league_league_name', 'season_league_league_slug', 'season_league_level_level_name', 'season_league_level_level_number', 'season_league_season_id', 'season_losses', 'season_manager', 'season_manager_manager_id', 'season_manager_manager_name', 'season_manager_manager_slug', 'season_points', 'season_points_against', 'season_points_for', 'season_rank', 'season_season', 'season_total_matches', 'season_wins', 'team_name']


In [35]:
# Map WC tournament -> TM season_id (year the season started)
WC_SEASON_ID = {
    "WC-2006": 2005,
    "WC-2010": 2009,
    "WC-2014": 2013,
    "WC-2018": 2017,
    "WC-2022": 2021,
}

# We need to join on: club_id (from performances) + competition_id + season_id
# First, get the club_id and competition_id for each player's primary club
# These come from the performances merge — let's pull them

# Re-extract club_id and primary_comp_id from the performances merge
merged_club_info = performances.merge(
    matched[["tm_player_id", "tournament_id", "target_season"]].rename(
        columns={"tournament_id": "wc_tournament"}
    ),
    left_on=["player_id", "season_name"],
    right_on=["tm_player_id", "target_season"],
    how="inner",
)
merged_club_info["domestic_league"] = merged_club_info["competition_id"].map(
    ALL_DOMESTIC
)

# Get primary domestic comp row per player (same logic as before)
dom_info = merged_club_info[merged_club_info["domestic_league"].notna()].sort_values(
    "nb_on_pitch", ascending=False
)
dom_info = dom_info.groupby(["tm_player_id", "wc_tournament"]).first().reset_index()
dom_info = dom_info[["tm_player_id", "wc_tournament", "team_id", "competition_id"]]
dom_info.columns = ["tm_player_id", "wc_tournament", "club_id", "domestic_comp_id"]

# Map season_id
dom_info["season_id"] = dom_info["wc_tournament"].map(WC_SEASON_ID)

print(f"Players with club info for ranking lookup: {len(dom_info)}")
dom_info.head()

Players with club info for ranking lookup: 2552


,tm_player_id,wc_tournament,club_id,domestic_comp_id,season_id
0,10,WC-2006,86,L1,2005
1,10,WC-2010,27,L1,2009
2,10,WC-2014,398,IT1,2013
3,14,WC-2006,409,A1,2005
4,26,WC-2014,16,L1,2013


In [36]:
# Join with team_competitions_seasons to get season_rank
tcs_slim = tcs[
    [
        "club_id",
        "competition_id",
        "season_id",
        "season_rank",
        "season_points",
        "season_goals_for",
        "season_goals_against",
        "season_goal_difference",
        "season_wins",
        "season_losses",
        "season_draws",
        "season_total_matches",
        "season_league_level_level_number",
    ]
].copy()

# club_id in performances is int, in tcs might be different — ensure same type
dom_info["club_id"] = dom_info["club_id"].astype(int)
tcs_slim["club_id"] = tcs_slim["club_id"].astype(int)
tcs_slim["season_id"] = tcs_slim["season_id"].astype(int)

club_season = dom_info.merge(
    tcs_slim,
    left_on=["club_id", "domestic_comp_id", "season_id"],
    right_on=["club_id", "competition_id", "season_id"],
    how="left",
)

# Rename for clarity
club_season = club_season.rename(
    columns={
        "season_rank": "club_league_position",
        "season_league_level_level_number": "league_tier",
        "season_points": "club_points",
        "season_goals_for": "club_gf",
        "season_goals_against": "club_ga",
        "season_goal_difference": "club_gd",
    }
)

club_season = club_season[
    [
        "tm_player_id",
        "wc_tournament",
        "club_league_position",
        "league_tier",
        "club_points",
        "club_gf",
        "club_ga",
        "club_gd",
    ]
]

final = final.merge(club_season, on=["tm_player_id", "wc_tournament"], how="left")

print(
    f"Club ranking coverage: {final['club_league_position'].notna().sum()}/{len(final)}"
)
print(f"League tier coverage: {final['league_tier'].notna().sum()}/{len(final)}")

print(f"\nSample:")
print(
    final[final["club_league_position"].notna()]
    .head(15)[
        [
            "family_name",
            "club",
            "league",
            "club_league_position",
            "league_tier",
            "wc_tournament",
        ]
    ]
    .to_string()
)

Club ranking coverage: 2461/2891
League tier coverage: 2461/2891

Sample:
   family_name                    club          league  club_league_position  league_tier wc_tournament
3    Mantorras              SL Benfica   Liga Portugal                   3.0          1.0       WC-2006
4       Mateus          Gil Vicente FC   Liga Portugal                  12.0          1.0       WC-2006
5      Marques               Hull City    Championship                  18.0          2.0       WC-2006
6       Flávio        FC Hansa Rostock   2. Bundesliga                  10.0          2.0       WC-2006
7       Buengo  Clermont Foot Auvergne         Ligue 2                  18.0          2.0       WC-2006
9    Coloccini  Deportivo de La Coruña          LaLiga                   8.0          1.0       WC-2006
10   Cambiasso             Inter Milan         Serie A                   1.0          1.0       WC-2006
11      Heinze       Manchester United  Premier League                   2.0          1.0     

## 11. League Country

In [ ]:
LEAGUE_COUNTRY = {
    "Premier League": ("England", "ENG"),
    "LaLiga": ("Spain", "ESP"),
    "Bundesliga": ("Germany", "GER"),
    "Serie A": ("Italy", "ITA"),
    "Ligue 1": ("France", "FRA"),
    "Eredivisie": ("Netherlands", "NED"),
    "Liga Portugal": ("Portugal", "POR"),
    "Super Lig": ("Turkey", "TUR"),
    "Premier Liga": ("Russia", "RUS"),
    "MLS": ("USA", "USA"),
    "JPL": ("Belgium", "BEL"),
    "Scottish Premiership": ("Scotland", "SCO"),
    "Super League Greece": ("Greece", "GRE"),
    "Ukrainian Premier Liga": ("Ukraine", "UKR"),
    "Super League Switzerland": ("Switzerland", "SUI"),
    "Austrian Bundesliga": ("Austria", "AUT"),
    "Danish Superliga": ("Denmark", "DEN"),
    "Ekstraklasa": ("Poland", "POL"),
    "Super Liga Serbia": ("Serbia", "SRB"),
    "1. HNL": ("Croatia", "CRO"),
    "Czech First League": ("Czech Republic", "CZE"),
    "Liga I Romania": ("Romania", "ROU"),
    "Slovak Super Liga": ("Slovakia", "SVK"),
    "NB I Hungary": ("Hungary", "HUN"),
    "Bulgarian First League": ("Bulgaria", "BUL"),
    "Cypriot First Division": ("Cyprus", "CYP"),
    "Allsvenskan": ("Sweden", "SWE"),
    "Eliteserien": ("Norway", "NOR"),
    "Ligat haAl": ("Israel", "ISR"),
    "BGL Ligue": ("Luxembourg", "LUX"),
    "Liga MX": ("Mexico", "MEX"),
    "Argentine Primera": ("Argentina", "ARG"),
    "Costa Rica Primera": ("Costa Rica", "CRC"),
    "Paraguay Primera": ("Paraguay", "PAR"),
    "Liga Nacional Honduras": ("Honduras", "HON"),
    "Liga Nacional Guatemala": ("Guatemala", "GUA"),
    "A-League": ("Australia", "AUS"),
    "J1 League": ("Japan", "JPN"),
    "K League 1": ("South Korea", "KOR"),
    "Saudi Pro League": ("Saudi Arabia", "KSA"),
    "Qatar Stars League": ("Qatar", "QAT"),
    "Persian Gulf League": ("Iran", "IRN"),
    "South African Premiership": ("South Africa", "RSA"),
    "Egyptian Premier League": ("Egypt", "EGY"),
    "Botola Pro": ("Morocco", "MAR"),
    "Ligue I Pro Tunisia": ("Tunisia", "TUN"),
    "NZ Football Championship": ("New Zealand", "NZL"),
    "Brasileirao": ("Brazil", "BRA"),
    "Championship": ("England", "ENG"),
    "League One": ("England", "ENG"),
    "League Two": ("England", "ENG"),
    "Scottish Championship": ("Scotland", "SCO"),
    "2. Bundesliga": ("Germany", "GER"),
    "Ligue 2": ("France", "FRA"),
    "Serie B": ("Italy", "ITA"),
    "LaLiga 2": ("Spain", "ESP"),
    "Liga Portugal 2": ("Portugal", "POR"),
    "Eerste Divisie": ("Netherlands", "NED"),
    "Belgian Second Division": ("Belgium", "BEL"),
    "Danish First Division": ("Denmark", "DEN"),
    "Turkish 1. Lig": ("Turkey", "TUR"),
}

country_full = {k: v[0] for k, v in LEAGUE_COUNTRY.items()}
country_code = {k: v[1] for k, v in LEAGUE_COUNTRY.items()}

final["league_country"] = final["home_league"].map(country_full)
mask = final["league_country"].isna() & final["league"].notna()
final.loc[mask, "league_country"] = final.loc[mask, "league"].map(country_full)

final["league_country_code"] = final["home_league"].map(country_code)
mask2 = final["league_country_code"].isna() & final["league"].notna()
final.loc[mask2, "league_country_code"] = final.loc[mask2, "league"].map(country_code)

print(f"League country coverage: {final['league_country'].notna().sum()}/{len(final)}")
print(final["league_country"].value_counts().head(10))

League country coverage: 2553/2891
league_country
England        572
Spain          323
Germany        322
Italy          307
France         216
Netherlands    102
Mexico          99
Portugal        92
Russia          78
Turkey          64
Name: count, dtype: int64


## 12. Export

In [38]:
final.to_csv("player_stats.csv", index=False)
# print(f"Saved: {final.shape}")
# print(f"\nColumns: {final.columns.tolist()}")